In [1]:
import pandas as pd
import numpy as np
import time

import sys
import os

import requests
import json
from pathlib import Path, PurePath
from datetime import datetime

c:\Users\MULINGWA STEPHEN\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [2]:
# Token for API access
token = "338f417f5f8ae25ba6eb01c878134153FE905CC0A125F5E827810F937A192125F15C8769"

In [3]:
def wialon_login(token, full=False):
    url = "https://hst-api.wialon.com/wialon/ajax.html"

    params = {
        "svc": "token/login",
        "params": json.dumps({"token": token})
    }

    response = requests.post(url, params=params)
    result = response.json()

    if full:
        return result
    return result.get("eid")

In [4]:
eid = wialon_login(token)
print("EID:", eid)

EID: 1075858b9cd480cdf1d3442d0c0124fb


In [5]:
import pandas as pd
from datetime import datetime, timezone

# Explicit UTC interval: 22 Feb 2026 07:00 → 23 Feb 2026 15:30
start_utc = datetime(2026, 4, 15, 0, 0, tzinfo=timezone.utc)
end_utc = datetime(2026, 4, 15, 23, 59, tzinfo=timezone.utc)

from_ts = int(start_utc.timestamp())
to_ts = int(end_utc.timestamp())

# 1) Execute speeding report for West Kenya (same parameters as Wialon UI)
exec_payload = {
    "svc": "report/exec_report",
    "params": json.dumps(
        {
            "reportResourceId": 17082202,
            "reportTemplateId": 220,
            "reportObjectId": 30182477,
            "reportObjectSecId": 0,
            "interval": {
                "flags": 0,
                "from": from_ts,
                "to": to_ts,
            },
        }
    ),
    "sid": eid,
}

exec_response = requests.post(
    "https://hst-api.wialon.com/wialon/ajax.html",
    data=exec_payload,
)
exec_result = exec_response.json()

# 2) Helper function to fetch any table by index
def fetch_table(report_tables, table_index):
    if table_index >= len(report_tables):
        print(f"Table index {table_index} not found. Only {len(report_tables)} table(s) available.")
        return pd.DataFrame()

    table_meta = report_tables[table_index]
    headers = table_meta.get("header", [])
    row_count = table_meta.get("rows", 0)

    rows_payload = {
        "svc": "report/get_result_rows",
        "params": json.dumps(
            {
                "tableIndex": table_index,   # <-- key change: use the target index
                "indexFrom": 0,
                "indexTo": max(row_count - 1, 0),
            }
        ),
        "sid": eid,
    }

    rows_response = requests.post(
        "https://hst-api.wialon.com/wialon/ajax.html",
        data=rows_payload,
    )
    rows_json = rows_response.json()

    rows_data = []
    for row in rows_json:
        cells = row.get("c", [])
        values = [c.get("t") if isinstance(c, dict) else c for c in cells]
        rows_data.append(values)

    if rows_data:
        max_cols = min(len(headers), len(rows_data[0]))
        return pd.DataFrame(rows_data, columns=headers[:max_cols])
    else:
        return pd.DataFrame(columns=headers)


report_tables = exec_result.get("reportResult", {}).get("tables", [])

# Fetch both sheets
ena_coach_summary  = fetch_table(report_tables, table_index=0)  # Sheet 1
ena_coach_ecodriving = fetch_table(report_tables, table_index=1)  # Sheet 2
ena_coach_location = fetch_table(report_tables, table_index=2)  # Sheet 3
ena_coach_fillings = fetch_table(report_tables, table_index=3)  # Sheet 4
ena_coach_drains = fetch_table(report_tables, table_index=4)  # Sheet 5
ena_coach_idling = fetch_table(report_tables, table_index=5)  # Sheet 6



In [6]:
ena_coach_idling

,Grouping,Initial location,Final location,Engine hours,In motion,Idling,Driver
0,ENA COACH - KDE 181Q,"New Station Road, Kisumu, Kenya","Kaplong-Narok-Maai Road, Kenya, 6.29 km from N...",15:51:21,11:49:25,4:01:56,
1,ENA COACH - KDE 182Q,"Nakuru - Eldoret Road, Kenya, Nuru Estate","Old Nairobi Road, Nakuru, Kenya",13:54:00,11:48:00,2:06:00,


In [13]:
ena_coach_fillings

,Grouping,Filling or charge time,Location,Initial fuel level,Filled,Final fuel level,Registered filling,Filling difference,Driver,Mileage
0,ENA COACH - KDE 181Q,15.04.2026 19:22:20,"Waiyaki Way, Nairobi, Kenya",248 l,348 l,400 l,0.00 l,348 l,,1247596 km
1,ENA COACH - KDE 182Q,-----,,-----,-----,-----,-----,0.00 l,,1269724 km


In [14]:
ena_coach_drains

,Grouping,Initial location,Drain time,Final location,Drained,Final fuel level,Driver,Mileage
0,ENA COACH - KDE 181Q,,-----,,-----,-----,,1247212 km
1,ENA COACH - KDE 182Q,,-----,,-----,-----,,1269724 km


In [ ]:
ena_coach_location

,Grouping,Last message time,Last coordinates time,Location,Driver
0,ENA COACH - KDE 181Q,16.04.2026 13:44:19,16.04.2026 13:44:19,"Murang'A Road, Nairobi, Kenya",
1,ENA COACH - KDE 182Q,16.04.2026 13:44:10,16.04.2026 13:44:10,"Obote Road, Kisumu, Kenya",


: 

In [6]:
ena_coach_summary

,Grouping,Mileage in all messages,Avg. speed,Max. speed,Engine hours,Avg. mileage per unit of fuel by FLS,Total fillings,Total drains,Filled,Drained,Consumed by AbsFCS,Avg. consumption by AbsFCS
0,ENA COACH - KDE 181Q,620 km,33 km/h,89 km/h,15:51:21,2.32 km,2,0,348 l,0.00 l,255 l,41 l/100 km
1,ENA COACH - KDE 182Q,619 km,26 km/h,94 km/h,13:54:00,2.28 km,0,0,0.00 l,0.00 l,209 l,34 l/100 km


In [7]:
def detailize_table(table_index, row_index, col_index=0):
    """
    Fetch detailization (subrows) for a specific row in a report table
    """
    detail_payload = {
        "svc": "report/get_result_subrows",
        "params": json.dumps(
            {
                "tableIndex": table_index,
                "rowIndex": row_index,
                "colIndex": col_index,  # usually 0 works, but depends on report structure
                "indexFrom": 0,
                "indexTo": 1000,  # adjust if you expect more rows
            }
        ),
        "sid": eid,
    }

    response = requests.post(
        "https://hst-api.wialon.com/wialon/ajax.html",
        data=detail_payload,
    )

    result = response.json()

    rows_data = []
    for row in result:
        cells = row.get("c", [])
        values = [c.get("t") if isinstance(c, dict) else c for c in cells]
        rows_data.append(values)

    return rows_data

In [8]:
all_details = []

table_index = 1  # ecodriving table

for i in range(len(ena_coach_ecodriving)):
    details = detailize_table(table_index=table_index, row_index=i)

    for d in details:
        all_details.append([i] + d)  # attach parent row index

# Convert to DataFrame
detail_df = pd.DataFrame(all_details)

num_cols = detail_df.shape[1]

detail_columns = ["parent_row"] + [f"field_{i}" for i in range(1, num_cols)]

detail_df.columns = detail_columns

In [9]:
# Drop unwanted columns
detail_df = detail_df.drop(columns=["parent_row", "field_1", "field_12"], errors="ignore")

# Keep only expected number of columns
detail_df = detail_df.iloc[:, :100000]

# Rename columns
detail_df.columns = [
    "Grouping",
    "Violation",
    "Beginning",
    "Initial location",
    "End",
    "Final location",
    "Avg. speed",
    "Max. speed",
    "Duration",
    "Mileage",
    "Count"
]

In [10]:
detail_df.head()

,Grouping,Violation,Beginning,Initial location,End,Final location,Avg. speed,Max. speed,Duration,Mileage,Count
0,ENA COACH - KDE 181Q,Over Speeding,15.04.2026 05:03:04,"New Station Road, Kisumu, Kenya",15.04.2026 23:58:56,"Kaplong-Narok-Maai Road, Kenya, 6.29 km from N...",33 km/h,89 km/h,18:55:52,620 km,1
1,ENA COACH - KDE 181Q,Harsh Cornering,15.04.2026 05:03:04,"New Station Road, Kisumu, Kenya",15.04.2026 05:03:16,"New Station Road, Kisumu, Kenya",0 km/h,0 km/h,0:00:12,0.00 km,1
2,ENA COACH - KDE 181Q,Harsh Cornering,15.04.2026 05:03:16,"New Station Road, Kisumu, Kenya",15.04.2026 14:15:46,"Haile Selassie Avenue, Nairobi, Kenya",39 km/h,88 km/h,9:12:30,362 km,1
3,ENA COACH - KDE 181Q,Harsh Braking,15.04.2026 06:23:29,"Kisumu-Nairobi Highway, Kisumu, Kenya",15.04.2026 06:23:29,"Kisumu-Nairobi Highway, Kisumu, Kenya",0 km/h,60 km/h,0:00:00,0.00 km,1
4,ENA COACH - KDE 181Q,Harsh Braking,15.04.2026 07:13:06,"Kericho-Kisumu Highway, Kenya, 2.77 km from Ka...",15.04.2026 07:13:07,"Kericho-Kisumu Highway, Kenya, 2.80 km from Ka...",126 km/h,58 km/h,0:00:01,0.04 km,1


In [11]:
unique_violations = detail_df['Violation'].unique().tolist()
unique_violations

['Over Speeding',
 'Harsh Cornering',
 'Harsh Braking',
 'Free Wheeling',
 'Over Revving']

In [12]:
from urllib.parse import quote_plus
from IPython.display import display, HTML


def make_location_hyperlink(location: str) -> str:
    """Return clickable Google Maps search link for a location string."""
    if pd.isna(location) or str(location).strip() == "":
        return ""
    loc = str(location).strip()
    url = f"https://www.google.com/maps/search/?api=1&query={quote_plus(loc)}"
    return f'<a href="{url}" target="_blank">{loc}</a>'


violation_order = [
    "Harsh Cornering",
    "Over Speeding",
    "Harsh Braking",
    "Free Wheeling",
    "Over Revving",
]

violation_tables = {}

for violation in violation_order:
    vdf = detail_df[detail_df["Violation"] == violation].copy()

    # Convert location columns to hyperlinks
    for col in ["Initial location", "Final location"]:
        if col in vdf.columns:
            vdf[col] = vdf[col].apply(make_location_hyperlink)

    violation_tables[violation] = vdf

In [13]:
harsh_cornering_df = detail_df[detail_df["Violation"] == "Harsh Cornering"].copy()
for col in ["Initial location", "Final location"]:
    harsh_cornering_df[col] = harsh_cornering_df[col].apply(make_location_hyperlink)
harsh_cornering_df.index = range(1, len(harsh_cornering_df) + 1)
HTML(harsh_cornering_df.head(5).to_html(escape=False))

,Grouping,Violation,Beginning,Initial location,End,Final location,Avg. speed,Max. speed,Duration,Mileage,Count
1,ENA COACH - KDE 181Q,Harsh Cornering,15.04.2026 05:03:04,"New Station Road, Kisumu, Kenya",15.04.2026 05:03:16,"New Station Road, Kisumu, Kenya",0 km/h,0 km/h,0:00:12,0.00 km,1
2,ENA COACH - KDE 181Q,Harsh Cornering,15.04.2026 05:03:16,"New Station Road, Kisumu, Kenya",15.04.2026 14:15:46,"Haile Selassie Avenue, Nairobi, Kenya",39 km/h,88 km/h,9:12:30,362 km,1
3,ENA COACH - KDE 181Q,Harsh Cornering,15.04.2026 14:15:53,"Haile Selassie Avenue, Nairobi, Kenya",15.04.2026 15:38:01,"Shimo La Tewa Road, Nairobi, Kenya",3 km/h,58 km/h,1:22:08,4.41 km,1
4,ENA COACH - KDE 181Q,Harsh Cornering,15.04.2026 15:38:01,"Shimo La Tewa Road, Nairobi, Kenya",15.04.2026 15:59:31,"Shimo La Tewa Road, Nairobi, Kenya",0 km/h,5 km/h,0:21:30,0.00 km,1
5,ENA COACH - KDE 181Q,Harsh Cornering,15.04.2026 15:59:53,"Shimo La Tewa Road, Nairobi, Kenya",15.04.2026 18:25:50,"Haile Selassie Avenue, Nairobi, Kenya",2 km/h,60 km/h,2:25:57,4.13 km,1


In [14]:
over_speeding_df = detail_df[detail_df["Violation"] == "Over Speeding"].copy()
for col in ["Initial location", "Final location"]:
    over_speeding_df[col] = over_speeding_df[col].apply(make_location_hyperlink)
over_speeding_df.index = range(1, len(over_speeding_df) + 1)
HTML(over_speeding_df.head(5).to_html(escape=False))

,Grouping,Violation,Beginning,Initial location,End,Final location,Avg. speed,Max. speed,Duration,Mileage,Count
1,ENA COACH - KDE 181Q,Over Speeding,15.04.2026 05:03:04,"New Station Road, Kisumu, Kenya",15.04.2026 23:58:56,"Kaplong-Narok-Maai Road, Kenya, 6.29 km from Ndamichoni",33 km/h,89 km/h,18:55:52,620 km,1
2,ENA COACH - KDE 182Q,Over Speeding,15.04.2026 00:00:00,"Nakuru-Kisumu Road, Nakuru, Kenya",15.04.2026 23:59:00,"Old Nairobi Road, Nakuru, Kenya",26 km/h,94 km/h,23:59:00,619 km,1


In [15]:
free_wheeling_df = detail_df[detail_df["Violation"] == "Free Wheeling"].copy()
for col in ["Initial location", "Final location"]:
    free_wheeling_df[col] = free_wheeling_df[col].apply(make_location_hyperlink)
free_wheeling_df.index = range(1, len(free_wheeling_df) + 1)
HTML(free_wheeling_df.head(5).to_html(escape=False))

,Grouping,Violation,Beginning,Initial location,End,Final location,Avg. speed,Max. speed,Duration,Mileage,Count
1,ENA COACH - KDE 181Q,Free Wheeling,15.04.2026 07:47:23,"Kericho-Kisumu Highway, Kericho, Kenya",15.04.2026 07:47:29,"Kericho-Kisumu Highway, Kericho, Kenya",60 km/h,46 km/h,0:00:06,0.10 km,1
2,ENA COACH - KDE 181Q,Free Wheeling,15.04.2026 08:20:06,"Kericho-Kisumu Highway, Kenya, Chepseon",15.04.2026 08:20:08,"Kericho-Kisumu Highway, Kenya, Chepseon",45 km/h,46 km/h,0:00:02,0.03 km,1
3,ENA COACH - KDE 181Q,Free Wheeling,15.04.2026 09:46:50,"Old Nairobi Road, Nakuru, Kenya",15.04.2026 09:46:59,"Old Nairobi Road, Nakuru, Kenya",42 km/h,38 km/h,0:00:09,0.10 km,1
4,ENA COACH - KDE 181Q,Free Wheeling,15.04.2026 11:45:06,"Nairobi - Nakuru Road, Kenya, Line",15.04.2026 11:45:11,"Nairobi - Nakuru Road, Kenya, Line",47 km/h,43 km/h,0:00:05,0.07 km,1
5,ENA COACH - KDE 181Q,Free Wheeling,15.04.2026 11:46:16,"Nairobi - Nakuru Road, Kenya, Line",15.04.2026 11:46:22,"Nairobi - Nakuru Road, Kenya, Line",57 km/h,50 km/h,0:00:06,0.10 km,1


In [16]:
over_revving_df = detail_df[detail_df["Violation"] == "Over Revving"].copy()
for col in ["Initial location", "Final location"]:
    over_revving_df[col] = over_revving_df[col].apply(make_location_hyperlink)
over_revving_df.index = range(1, len(over_revving_df) + 1)
HTML(over_revving_df.head(5).to_html(escape=False))

,Grouping,Violation,Beginning,Initial location,End,Final location,Avg. speed,Max. speed,Duration,Mileage,Count
1,ENA COACH - KDE 182Q,Over Revving,15.04.2026 01:27:35,"Kericho-Kisumu Highway, Kenya, 4.73 km from Kapkaim",15.04.2026 01:27:40,"Kericho-Kisumu Highway, Kenya, 4.71 km from Kapkaim",43 km/h,26 km/h,0:00:05,0.06 km,1


In [17]:
harsh_braking_df = detail_df[detail_df["Violation"] == "Harsh Braking"].copy()
for col in ["Initial location", "Final location"]:
    harsh_braking_df[col] = harsh_braking_df[col].apply(make_location_hyperlink)
harsh_braking_df.index = range(1, len(harsh_braking_df) + 1)
HTML(harsh_braking_df.head(5).to_html(escape=False))

,Grouping,Violation,Beginning,Initial location,End,Final location,Avg. speed,Max. speed,Duration,Mileage,Count
1,ENA COACH - KDE 181Q,Harsh Braking,15.04.2026 06:23:29,"Kisumu-Nairobi Highway, Kisumu, Kenya",15.04.2026 06:23:29,"Kisumu-Nairobi Highway, Kisumu, Kenya",0 km/h,60 km/h,0:00:00,0.00 km,1
2,ENA COACH - KDE 181Q,Harsh Braking,15.04.2026 07:13:06,"Kericho-Kisumu Highway, Kenya, 2.77 km from Kanyaruga",15.04.2026 07:13:07,"Kericho-Kisumu Highway, Kenya, 2.80 km from Kanyaruga",126 km/h,58 km/h,0:00:01,0.04 km,1
3,ENA COACH - KDE 181Q,Harsh Braking,15.04.2026 07:16:17,"Kericho-Kisumu Highway, Kenya, 5.17 km from Kiligis",15.04.2026 07:16:23,"Kericho-Kisumu Highway, Kenya, 5.24 km from Kiligis",42 km/h,24 km/h,0:00:06,0.07 km,1
4,ENA COACH - KDE 181Q,Harsh Braking,15.04.2026 07:32:43,"Kericho-Kisumu Highway, Kenya, 2.84 km from Kaitui",15.04.2026 07:32:43,"Kericho-Kisumu Highway, Kenya, 2.83 km from Kaitui",0 km/h,38 km/h,0:00:00,0.00 km,1
5,ENA COACH - KDE 181Q,Harsh Braking,15.04.2026 07:50:50,"Kericho-Kisumu Highway, Kericho, Kenya",15.04.2026 07:50:53,"Kericho-Kisumu Highway, Kericho, Kenya",48 km/h,44 km/h,0:00:03,0.04 km,1


In [18]:
def to_number(series, keep_float=True):
    """Extract first numeric value from text like '12796 km' or '35 l/100 km'."""
    extracted = series.astype(str).str.extract(r"([0-9]+(?:\.[0-9]+)?)", expand=False)
    nums = pd.to_numeric(extracted, errors="coerce")
    if keep_float:
        return nums
    return nums.fillna(0).astype(int)


# Build a normalized summary table from available report data
summary_df = ena_coach_summary.copy()
summary_df["Distance_km"] = to_number(summary_df["Mileage in all messages"])
summary_df["Average_speed_kmh"] = to_number(summary_df["Avg. speed"])
summary_df["Avg_fuel_km_per_l"] = to_number(summary_df["Avg. mileage per unit of fuel by FLS"])
summary_df["Fuel_consumption_diesel_l"] = to_number(summary_df["Consumed by AbsFCS"])
summary_df["Fuel_drains_qty"] = to_number(summary_df["Total drains"], keep_float=False)

# Parse grouping into Drivers and Vehicle where possible
split_grouping = summary_df["Grouping"].astype(str).str.split(" - ", n=1, expand=True)
summary_df["Drivers"] = split_grouping[0].str.strip()
summary_df["Vehicle"] = split_grouping[1].fillna(split_grouping[0]).str.strip()

# Aggregations from detail table
detail_work = detail_df.copy()
detail_work["Mileage_num"] = to_number(detail_work["Mileage"])

violation_counts = (
    detail_work.groupby(["Grouping", "Violation"]).size().unstack(fill_value=0)
)

freewheel_km = (
    detail_work[detail_work["Violation"] == "Free Wheeling"]
    .groupby("Grouping")["Mileage_num"]
    .sum()
)

# Required final columns (kept in exact order requested)
required_columns = [
    "Drivers",
    "Vehicle",
    "Average fuel consumption km/l",
    "Distance  km",
    "Engine running time  hours, minutes",
    "Brake applications  #/100",
    "Harsh brake applications  #/100",
    "Harsh acceleration  #/100",
    "Idling  % of engine running time",
    "Engine overspeed  % of engine running time",
    "Powertrain coasting  % of distance",
    "Fuel consumption - Diesel  litres",
    "Fuel consumption - idling - Diesel  litres",
    "Engine running time, idling  hours, minutes",
    "Average weight  tonnes",
    "Average speed  km/h",
    "Freewheel coasting  km",
    "Brake applications  Qty",
    "Harsh brake applications  Qty",
    "# of Overspeeding Incidents",
    "# Fuel Drains",
    "# Freewheeling",
]

final_rows = []
for _, row in summary_df.iterrows():
    grouping = row["Grouping"]
    distance = row["Distance_km"]

    harsh_brake_qty = int(violation_counts.loc[grouping, "Harsh Braking"]) if (
        grouping in violation_counts.index and "Harsh Braking" in violation_counts.columns
    ) else 0

    overspeed_qty = int(violation_counts.loc[grouping, "Over Speeding"]) if (
        grouping in violation_counts.index and "Over Speeding" in violation_counts.columns
    ) else 0

    freewheel_qty = int(violation_counts.loc[grouping, "Free Wheeling"]) if (
        grouping in violation_counts.index and "Free Wheeling" in violation_counts.columns
    ) else 0

    brake_per_100 = (harsh_brake_qty / distance * 100) if pd.notna(distance) and distance > 0 else np.nan
    harsh_brake_per_100 = brake_per_100

    final_rows.append(
        {
            "Drivers": row["Drivers"],
            "Vehicle": row["Vehicle"],
            "Average fuel consumption km/l": row["Avg_fuel_km_per_l"],
            "Distance  km": distance,
            "Engine running time  hours, minutes": row.get("Engine hours", np.nan),
            "Brake applications  #/100": brake_per_100,
            "Harsh brake applications  #/100": harsh_brake_per_100,
            "Harsh acceleration  #/100": np.nan,
            "Idling  % of engine running time": np.nan,
            "Engine overspeed  % of engine running time": np.nan,
            "Powertrain coasting  % of distance": np.nan,
            "Fuel consumption - Diesel  litres": row["Fuel_consumption_diesel_l"],
            "Fuel consumption - idling - Diesel  litres": np.nan,
            "Engine running time, idling  hours, minutes": np.nan,
            "Average weight  tonnes": np.nan,
            "Average speed  km/h": row["Average_speed_kmh"],
            "Freewheel coasting  km": float(freewheel_km.get(grouping, 0.0)),
            "Brake applications  Qty": harsh_brake_qty,
            "Harsh brake applications  Qty": harsh_brake_qty,
            "# of Overspeeding Incidents": overspeed_qty,
            "# Fuel Drains": int(row["Fuel_drains_qty"]) if pd.notna(row["Fuel_drains_qty"]) else 0,
            "# Freewheeling": freewheel_qty,
        }
    )

final_driver_vehicle_table = pd.DataFrame(final_rows, columns=required_columns)

# Optional formatting for easier reading
for c in [
    "Average fuel consumption km/l",
    "Distance  km",
    "Brake applications  #/100",
    "Harsh brake applications  #/100",
    "Fuel consumption - Diesel  litres",
    "Average speed  km/h",
    "Freewheel coasting  km",
]:
    final_driver_vehicle_table[c] = pd.to_numeric(final_driver_vehicle_table[c], errors="coerce").round(2)

with pd.option_context("display.max_columns", None, "display.width", 2000, "display.max_colwidth", None):
    display(final_driver_vehicle_table)

,Drivers,Vehicle,Average fuel consumption km/l,Distance km,"Engine running time hours, minutes",Brake applications #/100,Harsh brake applications #/100,Harsh acceleration #/100,Idling % of engine running time,Engine overspeed % of engine running time,Powertrain coasting % of distance,Fuel consumption - Diesel litres,Fuel consumption - idling - Diesel litres,"Engine running time, idling hours, minutes",Average weight tonnes,Average speed km/h,Freewheel coasting km,Brake applications Qty,Harsh brake applications Qty,# of Overspeeding Incidents,# Fuel Drains,# Freewheeling
0,ENA COACH,KDE 181Q,2.32,620,15:51:21,3.23,3.23,NaN,NaN,NaN,NaN,255,NaN,NaN,NaN,33,0.54,20,20,1,0,8
1,ENA COACH,KDE 182Q,2.28,619,13:54:00,0.00,0.00,NaN,NaN,NaN,NaN,209,NaN,NaN,NaN,26,0.35,0,0,1,0,5


In [ ]:
# Detailization for fuel tables (ena_coach_fillings and ena_coach_drains)
# Uses report/get_result_subrows exactly like ecodriving detailization flow.

def detailize_report_table(table_index, parent_df, col_index=0, max_rows=1000):
    all_details = []

    for i in range(len(parent_df)):
        detail_payload = {
            "svc": "report/get_result_subrows",
            "params": json.dumps(
                {
                    "tableIndex": table_index,
                    "rowIndex": i,
                    "colIndex": col_index,
                    "indexFrom": 0,
                    "indexTo": max_rows,
                }
            ),
            "sid": eid,
        }

        response = requests.post(
            "https://hst-api.wialon.com/wialon/ajax.html",
            data=detail_payload,
        )

        result = response.json()
        for row in result:
            cells = row.get("c", [])
            values = [c.get("t") if isinstance(c, dict) else c for c in cells]
            all_details.append(values)

    detail_df = pd.DataFrame(all_details)

    # If there are no subrows, keep original table as fallback
    if detail_df.empty:
        return parent_df.copy()

    # Align columns with parent dataframe headers when possible
    parent_cols = list(parent_df.columns)
    if len(detail_df.columns) >= len(parent_cols):
        detail_df = detail_df.iloc[:, :len(parent_cols)]
        detail_df.columns = parent_cols
    else:
        # If detailized rows have fewer fields, keep generic names for unmatched columns
        detail_df.columns = [f"field_{i}" for i in range(len(detail_df.columns))]

    return detail_df


# Build detailed dataframes for the two fuel tables
ena_coach_fillings = detailize_report_table(table_index=3, parent_df=ena_coach_fillings)
ena_coach_drains = detailize_report_table(table_index=4, parent_df=ena_coach_drains)

# Display detailized outputs
display(ena_coach_fillings.head())
display(ena_coach_drains.head())